# XTTS Embedding Extraction — Emilia Dataset Sample

Loads one audio sample from the Emilia dataset (streamed, no full download), then extracts the two XTTS conditioning embeddings:
- `gpt_cond_latent` — style/prosody latent fed into the GPT
- `speaker_embedding` — d-vector fed into the HiFiGAN decoder

In [1]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv()
login(token=os.getenv("HUGGINGFACE_TOKEN"))

In [2]:
from datasets import load_dataset

# Stream one English sample — no full download
ds = load_dataset(
    "amphion/Emilia-Dataset",
    split="train",
    streaming=True,
)
sample = next(iter(ds))

print("Keys:", list(sample.keys()))
print("Text:", sample.get("text", sample.get("json", {}).get("text", "—")))

audio = sample["mp3"]  # dict with 'array' and 'sampling_rate'
print(
    f"Sample rate: {audio['sampling_rate']} Hz, length: {len(audio['array'])} samples"
)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'amphion/Emilia-Dataset' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Resolving data files:   0%|          | 0/4343 [00:00<?, ?it/s]

Keys: ['json', 'mp3', '__key__', '__url__']
Text:  So. Chloe hat gesagt, ich soll noch unten gehen. Was ich natürlich auch machen werde.
Sample rate: 24000 Hz, length: 193968 samples


In [3]:
import torch

audio_tensor = torch.tensor(audio["array"]).unsqueeze(0).float()
sr = audio["sampling_rate"]
print(f"Audio tensor: {audio_tensor.shape}, sr={sr}")

Saved to: /var/folders/yx/5wp84kq17050h0n4rj2drhxm0000gn/T/tmp2k_z9v16.wav


In [ ]:
from TTS.api import TTS

# Downloads and caches to ~/.local/share/tts/ on first run
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
model = tts.synthesizer.tts_model
print("Model loaded:", type(model).__name__)

In [ ]:
gpt_cond_latent = model.get_gpt_cond_latents(
    audio_tensor, sr, length=model.config.gpt_cond_len
)
speaker_embedding = model.get_speaker_embedding(audio_tensor, sr)

print("gpt_cond_latent shape:", gpt_cond_latent.shape)
print("speaker_embedding shape:", speaker_embedding.shape)

In [ ]:
print("--- gpt_cond_latent ---")
print(f"  dtype:  {gpt_cond_latent.dtype}")
print(f"  min:    {gpt_cond_latent.min().item():.4f}")
print(f"  max:    {gpt_cond_latent.max().item():.4f}")
print(f"  mean:   {gpt_cond_latent.mean().item():.4f}")

print("\n--- speaker_embedding ---")
print(f"  dtype:  {speaker_embedding.dtype}")
print(f"  shape:  {speaker_embedding.shape}")
print(
    f"  l2norm: {torch.norm(speaker_embedding).item():.4f}"
)  # should be ~1.0 (L2-normalised)